##Analise de 'chamados'

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"


df_chamados = spark.table(f"{bronze_schema}.chamados")


In [0]:
display(df_chamados)

In [0]:
# ---------- auxiliares ----------
TOKENS_NULOS = ["NULL", "NONE", "NAN", "N/A"]      # textos que na prática significam "sem valor"

def texto(c):
    """Coluna como string, sem espaços nas pontas."""
    return F.trim(F.col(c).cast("string"))

def eh_faltante(c):
    """Null de verdade, string vazia/só espaços, ou texto tipo 'NULL'."""
    return F.col(c).isNull() | (texto(c) == "") | F.upper(texto(c)).isin(TOKENS_NULOS)

def mostrar(df_, descricao):
    """Só imprime a descrição e faz o display se o DataFrame tiver linhas."""
    if not df_.isEmpty():
        print(f"\n{descricao}")
        display(df_)

total = df_chamados.count()
print(f"Total de linhas: {total}")

# ---------- 1. duplicidade ----------
print("\n=== 1. DUPLICIDADE ===")
colunas_negocio = [c for c in df_chamados.columns if c != "ingestion_datetime"]
distintas = df_chamados.select(*colunas_negocio).distinct().count()
print(f"Linhas excedentes por cópia exata: {total - distintas}")

ids_repetidos = (
    df_chamados.filter(~eh_faltante("id_chamado"))
      .groupBy(texto("id_chamado").alias("id_chamado"))
      .agg(
          F.count("*").alias("ocorrencias"),
          F.size(F.collect_set(F.struct(*[c for c in colunas_negocio if c != "id_chamado"]))).alias("versoes_distintas"),
      )
      .filter(F.col("ocorrencias") > 1)
)
n_ids = ids_repetidos.count()
linhas_envolvidas = ids_repetidos.agg(F.sum("ocorrencias")).first()[0] or 0
n_divergentes = ids_repetidos.filter(F.col("versoes_distintas") > 1).count()
print(f"id_chamado repetidos: {n_ids} ids em {linhas_envolvidas} linhas ({linhas_envolvidas - n_ids} linhas excedentes)")
print(f"  -> {n_divergentes} desses ids aparecem com conteúdo diferente entre as linhas")
mostrar(ids_repetidos.orderBy(F.desc("ocorrencias"), "id_chamado"),
        "ids de chamado repetidos (ocorrencias = nº de linhas; versoes_distintas = nº de conteúdos diferentes)")

# ---------- 2. faltantes por coluna ----------
print("\n=== 2. ENTRADAS FALTANTES POR COLUNA ===")
exprs = []
for c in df_chamados.columns:
    exprs += [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}__nulo"),
        F.sum(F.when(texto(c) == "", 1).otherwise(0)).alias(f"{c}__vazio"),
        F.sum(F.when(F.upper(texto(c)).isin(TOKENS_NULOS), 1).otherwise(0)).alias(f"{c}__texto_nulo"),
    ]
r = df_chamados.agg(*exprs).first()

df_chamados_faltantes = (
    spark.createDataFrame(
        [(c, r[f"{c}__nulo"], r[f"{c}__vazio"], r[f"{c}__texto_nulo"]) for c in df_chamados.columns],
        "coluna string, nulo long, vazio long, texto_nulo long",
    )
    .withColumn("total_faltante", F.col("nulo") + F.col("vazio") + F.col("texto_nulo"))
    .withColumn("pct_faltante", F.round(F.col("total_faltante") * 100 / total, 2))
    .orderBy(F.desc("pct_faltante"))
)
mostrar(df_chamados_faltantes,
        "Faltantes por coluna: null real, string vazia e texto tipo 'NULL', com % sobre o total de linhas")

# ---------- 3. id_cliente x padrão de CPF ----------
print("\n=== 3. id_cliente x PADRÃO DE CPF (11 dígitos) ===")
tipo = dict(df_chamados.dtypes)["id_cliente"]
if tipo != "string":
    print(f"ATENÇÃO: id_cliente está como {tipo}. Se for numérico, CPFs que começam com 0 "
          "perderam o zero à esquerda e vão cair em 'menos de 11 dígitos'.")

cpf = texto("id_cliente")
df_chamados_cpf = df_chamados.withColumn(
    "categoria_cpf",
    F.when(eh_faltante("id_cliente"), "faltante")
     .when(cpf.rlike(r"^\d{11}$"), "ok: 11 dígitos")
     .when(cpf.rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"), "formatado com . e -")
     .when(cpf.rlike(r"^\d{1,10}$"), "menos de 11 dígitos")
     .when(cpf.rlike(r"^\d{12,}$"), "mais de 11 dígitos")
     .otherwise("caracteres inválidos"),
)
mostrar(
    df_chamados_cpf.groupBy("categoria_cpf").count()
          .withColumn("pct", F.round(F.col("count") * 100 / total, 2))
          .orderBy(F.desc("count")),
    "Distribuição de id_cliente por categoria de padrão de CPF",
)
mostrar(
    df_chamados_cpf.filter(~F.col("categoria_cpf").isin("ok: 11 dígitos", "faltante"))
          .groupBy("categoria_cpf", "id_cliente").count()
          .orderBy("categoria_cpf", F.desc("count")),
    "Valores de id_cliente fora do padrão, por categoria",
)

# ---------- 4. valores distintos em colunas categóricas ----------
print("\n=== 4. VALORES DISTINTOS: motivo, canal, resolvido ===")
for c in ["motivo", "canal", "resolvido"]:
    contagem = df_chamados.groupBy(c).count().orderBy(F.desc("count"))
    n_distintos = contagem.count()
    mostrar(contagem, f"'{c}': {n_distintos} valores distintos (grafia original) e nº de linhas de cada")

# ---------- 5. tempos em segundos ----------
print("\n=== 5. TEMPOS (segundos): inteiro não negativo + 10 maiores ===")
for c in ["tempo_espera", "tempo_atendimento"]:
    t = texto(c)
    df_chamados_t = df_chamados.withColumn(
        "categoria",
        F.when(eh_faltante(c), "faltante")
         .when(t.rlike(r"^\d+$"), "ok: inteiro >= 0")
         .when(t.rlike(r"^-\d+([.,]\d+)?$"), "negativo")
         .when(t.rlike(r"^\d+[.,]\d+$"), "decimal")
         .otherwise("não numérico"),
    )
    print(f"\n--- {c} ---")
    mostrar(
        df_chamados_t.groupBy("categoria").count()
            .withColumn("pct", F.round(F.col("count") * 100 / total, 2))
            .orderBy(F.desc("count")),
        f"Distribuição de {c} por categoria de formato",
    )
    mostrar(
        df_chamados_t.filter(~F.col("categoria").isin("ok: inteiro >= 0", "faltante"))
            .groupBy("categoria", c).count()
            .orderBy("categoria", F.desc("count")),
        f"Valores inconsistentes de {c} (negativos, decimais ou não numéricos)",
    )
    mostrar(
        df_chamados_t.filter(F.col("categoria") == "ok: inteiro >= 0")
            .withColumn("segundos", t.cast("long"))
            .select("id_chamado", "canal", "segundos",
                    (F.round(F.col("segundos") / 60, 1)).alias("minutos"),
                    (F.round(F.col("segundos") / 3600, 2)).alias("horas"))
            .orderBy(F.desc("segundos"))
            .limit(10),
        f"10 maiores valores válidos de {c}",
    )

# ---------- 6. id_atendente faltante x canal ----------
print("\n=== 6. id_atendente FALTANTE x canal ===")
n_sem_atendente = df_chamados.filter(eh_faltante("id_atendente")).count()
print(f"Linhas com id_atendente faltante: {n_sem_atendente} ({round(n_sem_atendente * 100 / total, 2)}%)")

sem_atendente_por_canal = (
    df_chamados.groupBy("canal")
      .agg(
          F.sum(F.when(eh_faltante("id_atendente"), 1).otherwise(0)).alias("sem_atendente"),
          F.count("*").alias("total_canal"),
      )
      .filter(F.col("sem_atendente") > 0)
      .withColumn("pct_do_canal", F.round(F.col("sem_atendente") * 100 / F.col("total_canal"), 2))
      .orderBy(F.desc("sem_atendente"))
)
mostrar(sem_atendente_por_canal,
        "Canais das linhas sem id_atendente (sem_atendente = linhas sem atendente no canal; "
        "pct_do_canal = % das linhas daquele canal que estão sem atendente)")

# ---------- 7. linhas com canal faltante ----------
print("\n=== 7. LINHAS COM canal FALTANTE ===")
mostrar(df_chamados.filter(eh_faltante("canal")),
        "Linhas em que canal é null, vazio ou texto tipo 'NULL'")

In [0]:
linhas_repetidas = (
    df_chamados
      .join(ids_repetidos.select("id_chamado"), on="id_chamado", how="inner")
      .orderBy("id_chamado")
)
display(linhas_repetidas)